# 🎙️ Whisper — Tunisian Arabic ASR Benchmark

**Model:** `openai/whisper-large-v3`  
**Config key:** `whisper_large_v3`

## 1 · Install & Imports

In [ ]:
!pip install -q transformers datasets torchaudio evaluate jiwer pyyaml


[notice] A new release of pip is available: 24.0 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
# Local path to benchmarking package
import sys
from pathlib import Path

BENCHMARK_ROOT = Path('/home/ala/dataset')
assert BENCHMARK_ROOT.exists(), f'Missing path: {BENCHMARK_ROOT}'

if str(BENCHMARK_ROOT) not in sys.path:
    sys.path.insert(0, str(BENCHMARK_ROOT))

print('Using benchmark root:', BENCHMARK_ROOT)

Using benchmark root: /home/ala/dataset


In [ ]:
from benchmark_utils import (
    load_config, get_device, print_gpu_info, setup_output_dir,
    load_benchmark, split_benchmark,
    compute_metrics, per_sample_wer,
    run_pipeline_inference, build_results_df,
    run_labelled_splits, run_unlabelled_splits,
    display_preview, display_worst, display_bulk_predictions,
    audio_inspector, display_summary, plot_wer_cer,
)
import numpy as np
import torch
import time
from pathlib import Path
from tqdm.auto import tqdm
from datasets import Audio as HFAudio

cfg = load_config(str(BENCHMARK_ROOT / 'config.yaml'))

# Override Colab-only paths for local execution
cfg['paths']['dataset'] = str(BENCHMARK_ROOT)
cfg['paths']['output_root'] = str(Path('/home/ala/TunisianDialogSystem/outputs/asr_benchmark_results'))

TARGET_SR    = cfg['evaluation']['target_sr']
TOP_N_WORST  = cfg['evaluation']['top_n_worst']
PREVIEW_ROWS = cfg['evaluation']['preview_rows']
RESUME = False

print('Dataset path:', cfg['paths']['dataset'])
print('Output root :', cfg['paths']['output_root'])

Dataset path: /home/ala/dataset
Output root : /home/ala/TunisianDialogSystem/outputs/asr_benchmark_results


## 2 · GPU Check

In [ ]:
device = get_device()
print_gpu_info()
torch_dtype = torch.float16 if device == 'cuda' else torch.float32
print(f'torch_dtype : {torch_dtype}')

Device : cuda
GPU    : NVIDIA GB10
VRAM   : 128.5 GB total  |  128.5 GB free
torch_dtype : torch.float16


## 3 · Load Model & Pipeline

### 3.1 Load processor and model

Whisper is a **seq2seq encoder-decoder** model. We force Arabic generation
via `language` and `task` in `generate_kwargs`.

> **Fix — duplicate logits processors:** We pass `language` and `task` only
> through `generate_kwargs` in the pipeline constructor, and explicitly set
> them on the generation config **before** calling `from_pretrained` so that
> `SuppressTokensLogitsProcessor` and `SuppressTokensAtBeginLogitsProcessor`
> are not duplicated at call time.

In [ ]:
from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq, pipeline

mcfg       = cfg['models']['whisper_large_v3']
MODEL_ID   = mcfg['model_id']
BATCH_SIZE = mcfg['batch_size']
OUTPUT_DIR = setup_output_dir(cfg, 'whisper_large_v3')

print(f'Loading {MODEL_ID} ...')
processor = AutoProcessor.from_pretrained(MODEL_ID)

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    MODEL_ID,
    torch_dtype=torch_dtype,
    low_cpu_mem_usage=True,
)

# Ensure forced_decoder_ids is cleared; the pipeline sets it via generate_kwargs
model.generation_config.forced_decoder_ids = None

model.to(device)
model.eval()

pipe = pipeline(
    'automatic-speech-recognition',
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    torch_dtype=torch_dtype,
    device=device,
    chunk_length_s=mcfg['chunk_length_s'],
    stride_length_s=mcfg['stride_length_s'],
      generate_kwargs={
        'language': mcfg['language'],
        'task':     mcfg['task'],
    },
)
print(f'✓ {MODEL_ID} loaded on {device} ({torch_dtype})')

Loading openai/whisper-large-v3 ...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[transformers] Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


✓ openai/whisper-large-v3 loaded on cuda (torch.float16)


## 4 · Mount Drive & Load Benchmark

In [ ]:
from datasets import DatasetDict, load_from_disk

# Build a DatasetDict from available on-disk split folders only
split_dirs = sorted([p for p in BENCHMARK_ROOT.iterdir() if p.is_dir()])
benchmark = DatasetDict()

for p in split_dirs:
    try:
        benchmark[p.name] = load_from_disk(str(p))
    except Exception:
        pass

print(f"Loaded splits: {list(benchmark.keys())}")
LABELLED_SPLITS, UNLABELLED_SPLITS = split_benchmark(benchmark) 

Loaded splits: ['labeled_algerian', 'labeled_linagora_cs_arabize', 'labeled_linagora_raw', 'unlabeled_youtube']
Labelled splits   : ['labeled_algerian', 'labeled_linagora_cs_arabize', 'labeled_linagora_raw']
Unlabelled splits : ['unlabeled_youtube']


## 5 · Inference

In [ ]:
def infer_fn(ds):
    return run_pipeline_inference(ds, pipe, BATCH_SIZE, TARGET_SR)

all_result_dfs, summary_rows = run_labelled_splits(
    benchmark,LABELLED_SPLITS, infer_fn,
    OUTPUT_DIR, 'whisper_largev3',
    PREVIEW_ROWS, TOP_N_WORST,
)


  Running: labeled_linagora_raw  (5482 samples)


Inferring:   0%|          | 0/1371 [00:00<?, ?batch/s]

[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> to see related `.generate()` flags.
[

## 6 · Per-sample preview

In [ ]:
display_preview(all_result_dfs, PREVIEW_ROWS)

## 7 · Worst predictions

In [ ]:
display_worst(all_result_dfs, TOP_N_WORST)

## 8 · Unlabelled inspection

In [ ]:
unlabelled_result_dfs = run_unlabelled_splits(
    benchmark, UNLABELLED_SPLITS, infer_fn, OUTPUT_DIR, 'whisper_largev3'
)

In [ ]:
audio_inspector(
    benchmark,
    unlabelled_result_dfs,
    UNLABELLED_SPLITS,
    target_sr=TARGET_SR,
)

In [ ]:
display_bulk_predictions(unlabelled_result_dfs)

## 9 · Summary

In [ ]:
summary_df = display_summary(
    summary_rows, OUTPUT_DIR, 'whisper_largev3', 'openai/whisper-large-v3'
)

In [ ]:
if summary_df is not None:
    plot_wer_cer(summary_df, OUTPUT_DIR, 'whisper_largev3', 'openai/whisper-large-v3')